In [2]:
from pyspark import SparkConf, SparkContext
import findspark
import json
from pyspark.sql import SparkSession

# 初始化 Spark 环境
findspark.init()

# 配置 Spark，设置应用名称和运行模式，分配足够的驱动内存
conf = SparkConf().setAppName('rdd1').setMaster('local[*]')
conf.set("spark.driver.memory", "4g")
sc = SparkContext(conf=conf)

# 打印当前 SparkContext 信息，用于确认配置
print(sc)

# 读取 JSON 格式的数据文件
rdd1 = sc.textFile('arxiv-metadata-oai-snapshot.json')
rdd1 = rdd1.map(lambda x:json.loads(x)) 

# 持久化 RDD 到内存
rdd1.persist()

# 打印当前默认的并行级别（默认分区数）
print(sc.defaultParallelism)

# 获取 RDD 的分区数
rdd1.getNumPartitions()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/05/17 09:05:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=rdd1>
1


99

In [34]:
import re
from pyspark.sql.functions import col,month
# 打印 RDD 中的记录总数
print(rdd1.count())

# 打印 RDD 中的前两条记录
print(rdd1.take(2))

# 提取 RDD 中所有记录的键名，去除重复项后收集为一个列表
print(rdd1.flatMap(lambda x: x.keys()).distinct().collect())

# 从每条记录中提取 'license' 字段的值，去除重复项后收集为一个列表
print(rdd1.map(lambda x: x['license']).distinct().collect())

# 计算并打印出具有最短和最长标题的记录的标题
shortest_title = rdd1.min(lambda x: len(x['title']))
longest_title = rdd1.max(lambda x: len(x['title']))
print("The Shortest title:", shortest_title['title'])
print("The Longest title:", longest_title['title'])

# 使用正则表达式从摘要中提取并返回括号内的缩写，计算并打印包含有效缩写的记录数量和前五条这样的记录
def get_abbreviations(line):
    result = re.search(r"\(([A-Za-z][^_ /\\<>]{5,})\)", line)
    if result:
        return result.group(1)

print("包含缩写的记录个数", rdd1.filter(lambda x: get_abbreviations(x['abstract']) is not None).count())
print("包含缩写的记录", rdd1.filter(lambda x: get_abbreviations(x["abstract"])).take(5))

# 利用 SparkSession 创建 DataFrame，并将 'update_date' 列的数据类型转换为日期类型，按月分组统计记录数量，并按月份排序后打印结果
spark = SparkSession(sc)

df = spark.createDataFrame(rdd1)
df = df.withColumn('update_date', col('update_date').cast('date'))
monthly_records = df.withColumn('month', month('update_date')).groupBy('month').count()
sorted_monthly_records = monthly_records.orderBy('month')
result = sorted_monthly_records.rdd.map(lambda row: (row['month'], row['count'])).collect()
# 提取每条记录中的页数信息（如果有的话），计算总页数、记录数和平均页数后打印平均页数
def extract_pages(record):
    comments = record.get('comments', '')
    if comments is not None:  # 确保 comments 不是 None
        pages_match = re.search(r'(\d+)\s+pages', comments)
        if pages_match:
            return int(pages_match.group(1))
    return None
    
pages_rdd = rdd1.map(extract_pages).filter(lambda x: x is not None)
total_pages = pages_rdd.reduce(lambda a, b: a + b)
num_records = pages_rdd.count()
average_pages = total_pages / num_records
print("the average of pages is:", average_pages)
sc.stop()

2011231


24/05/17 10:44:29 WARN BlockManager: Task 3199 already completed, not releasing lock for rdd_2_0


[{'id': '0704.0001', 'submitter': 'Pavel Nadolsky', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'comments': '37 pages, 15 figures; published version', 'journal-ref': 'Phys.Rev.D76:013009,2007', 'doi': '10.1103/PhysRevD.76.013009', 'report-no': 'ANL-HEP-PR-07-12', 'categories': 'hep-ph', 'license': None, 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from the Fermila

['authors_parsed', 'update_date', 'license', 'categories', 'authors', 'versions', 'submitter', 'title', 'doi', 'report-no', 'abstract', 'journal-ref', 'id', 'comments']


[None, 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'http://creativecommons.org/licenses/by-sa/4.0/', 'http://creativecommons.org/licenses/by/3.0/', 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'http://creativecommons.org/publicdomain/zero/1.0/', 'http://creativecommons.org/licenses/by/4.0/', 'http://creativecommons.org/licenses/by-nc-sa/4.0/', 'http://creativecommons.org/licenses/publicdomain/', 'http://creativecommons.org/licenses/by-nc-sa/3.0/']


The Shortest title: Q
The Longest title: Investigation of the 2-body system with a rotating central body (e. g.
  earth-moon system) within the Projective Unified Field theory: the transfer
  of rotational angular momentum and energy from the central body to the
  orbital 2-body system, the tidal and the non-tidal influences (mechanical,
  general-relativistic Lense-Thirring effect and cosmological
  PUFT-contributions)


包含缩写的记录个数 192721


24/05/17 10:50:48 WARN BlockManager: Task 3893 already completed, not releasing lock for rdd_2_0


包含缩写的记录 [{'id': '0704.0055', 'submitter': 'Per Hyldgaard', 'authors': 'Eleni Ziambaras, Jesper Kleis, Elsebeth Schroder, and Per Hyldgaard', 'title': 'Potassium intercalation in graphite: A van der Waals density-functional\n  study', 'comments': '10 pages, 5 figures', 'journal-ref': None, 'doi': '10.1103/PhysRevB.76.155425', 'report-no': None, 'categories': 'cond-mat.soft cond-mat.mtrl-sci', 'license': None, 'abstract': '  Potassium intercalation in graphite is investigated by first-principles\ntheory. The bonding in the potassium-graphite compound is reasonably well\naccounted for by traditional semilocal density functional theory (DFT)\ncalculations. However, to investigate the intercalate formation energy from\npure potassium atoms and graphite requires use of a description of the graphite\ninterlayer binding and thus a consistent account of the nonlocal dispersive\ninteractions. This is included seamlessly with ordinary DFT by a van der Waals\ndensity functional (vdW-DF) approach [

the average of pages is: 17.865863025605005
